In [1]:
import pandas as pd
import pyreadstat
# set pandas display options to show all columns
pd.set_option('display.max_columns', None)

# import basics
import pandas as pd
import numpy as np

# import tools
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

# import models
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from lightgbm import LGBMRegressor, LGBMClassifier
from xgboost import XGBRegressor, XGBClassifier
from tabpfn import TabPFNClassifier, TabPFNRegressor

# import viz
import altair as alt
alt.renderers.enable('mimetype') # for altair plots to be properly rendered on GH


RendererRegistry.enable('mimetype')

In [2]:
10 / 1.2


8.333333333333334

In [3]:
df_school, meta_school = pyreadstat.read_sav('../data/Data_PIRLS16(sav)/P4_SCHOOL16.sav')
df_student, meta_student = pyreadstat.read_sav('../data/Data_PIRLS16(sav)/P4_STUDENT16.sav')
df_teacher, meta_teacher = pyreadstat.read_sav('../data/Data_PIRLS16(sav)/P4_TEACHER16.sav')
df_link, meta_link = pyreadstat.read_sav('../data/Data_PIRLS16(sav)/P4_STD_TCH_LINK16.sav')


In [4]:
# cols to generate y
y_gen_cols = ['ASRREA01', 'ASRREA02', 'ASRREA03', 'ASRREA04', 'ASRREA05']
y_bin_cols = ['ASRIBM01', 'ASRIBM02', 'ASRIBM03', 'ASRIBM04', 'ASRIBM05']
# cols to drop because they might lead to leakage
cols_drop = y_gen_cols + ['ASRLIT01', 'ASRLIT02', 'ASRLIT03', 'ASRLIT04', 'ASRLIT05', 'ASRINF01', 'ASRINF02', 'ASRINF03', 'ASRINF04', 'ASRINF05', 'ASRIIE01', 'ASRIIE02', 'ASRIIE03', 'ASRIIE04', 'ASRIIE05', 'ASRRSI01', 'ASRRSI02', 'ASRRSI03', 'ASRRSI04', 'ASRRSI05', 'ASRIBM01', 'ASRIBM02', 'ASRIBM03', 'ASRIBM04', 'ASRIBM05', ]

In [5]:
def create_y(df, y_gen_cols):
    """Create y column by averaging the columns in y_gen_cols"""
    df['y'] = df[y_gen_cols].mean(axis=1)
    return df

def create_y_bin(df, y_bin_cols):
    """Create y column by averaging the columns in y_bin_cols"""
    df['y'] = df[y_bin_cols].mean(axis=1)
    # convert to binary
    df['y'] = df['y'].apply(lambda x: 1 if x >= 3 else 0)
    return df

def remove_cols(df, cols_drop):
    """Remove columns from the dataframe if they exist"""
    # drop column if it exists
    to_drop = [col for col in cols_drop if col in df.columns]
    # drop columns
    df = df.drop(columns=to_drop)
    return df

def merge_dfs(df_student, df_new, on):
    """ Merge df_new into df_student on the given column"""
    # record original number of rows
    original_rows = df_student.shape[0]
    # drop overlapping columns in df_new except for the merge column
    cols_to_drop = [col for col in df_new.columns if col in df_student.columns and col != on]
    df_new = df_new.drop(columns=cols_to_drop)
    # merge the dataframes
    df_student = df_student.merge(df_new, on=on, how='left', suffixes=('', '_new'))
    # drop the new columns that are now duplicates
    cols_to_drop = [col for col in df_student.columns if col.endswith('_new')]
    df_student = df_student.drop(columns=cols_to_drop)
    # assert that the number of rows is the same
    assert df_student.shape[0] == original_rows, f"Number of rows changed from {original_rows} to {df_student.shape[0]}"
    return df_student


In [6]:
def create_X_and_y(df_student, df_teacher, df_school, df_link, mode='binary'):
    """ Create a dataframe with the relevant columns from the student, teacher, school, and link dataframes"""
    global y_gen_cols, cols_drop, y_bin_cols
    if mode == 'binary':
        df_student = create_y_bin(df_student, y_bin_cols)
    else:
        df_student = create_y(df_student, y_gen_cols)
    df_student = remove_cols(df_student, cols_drop)
    df_school = remove_cols(df_school, cols_drop)
    df_teacher = remove_cols(df_teacher, cols_drop)
    df_link = remove_cols(df_link, cols_drop)
    
    df = merge_dfs(df_student, df_school, on='IDSCHOOL')
    df = merge_dfs(df, df_link, on='IDSTUD')
    df = merge_dfs(df, df_teacher, on='IDTEALIN')
    
    y = df['y']
    X = df.drop(columns=['y'])
    return X, y

In [7]:
X, y = create_X_and_y(df_student, df_teacher, df_school, df_link)
X.shape, y.shape

((4425, 385), (4425,))

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.3,
                                                    random_state=9527)

In [9]:
drop_cols = []
# create the list of cols to drop based on missing rate
missing_rates = (X_train.isnull().sum()/X_train.shape[0]).sort_values(ascending=False)[:20]
miss_rate_bar = 0.8
missing_cols = missing_rates[missing_rates > miss_rate_bar].index.tolist()
drop_cols += missing_cols

# id columns
id_cols = [col for col in X_train.columns if 'ID' in col]
drop_cols += id_cols

# date column
drop_cols += ['ITDATE']

# get list of columns that have only one unique value and with no missing values
def get_constant_cols(X):
    """ Get the constant columns from the dataframe"""
    const_cols = X.nunique()[X.nunique() == 1].index.tolist()
    # get the columns that have no missing values
    const_cols = [col for col in const_cols if X[col].isnull().sum() == 0]
    return const_cols

const_cols = get_constant_cols(X_train)
drop_cols += const_cols

# X_train.drop(columns=missing_cols, inplace=True)

# a custom sklearn pipeline function step to remove columns that are missing too much data
class ColumnDropper(BaseEstimator, TransformerMixin):
    def __init__(self, drop_cols):
        self.drop_cols = drop_cols
        self.is_fitted = False
        pass
    
    def fit(self, X, y=None):
        self.is_fitted = True
        return self

    def transform(self, X):
        assert self.is_fitted, "The ColumnDropper has not been fitted yet. Please call fit() before transform()."
        X_drop = X.drop(columns=self.drop_cols)
        self.remaining_cols = X_drop.columns
        return X_drop

In [10]:
models = {"Dummy": DummyRegressor(), 
        #  "LinearReg": LinearRegression(),
         "DT": DecisionTreeRegressor(),
        #  "RF": RandomForestRegressor(),
        #  "SVM": SVR(),
         "XGB": XGBRegressor(),
         "LGBM": LGBMRegressor(),
        "TabPFN": TabPFNRegressor(),
         }

models_bin = {"Dummy": DummyClassifier(),
              "DT": DecisionTreeClassifier(),
              "XGB": XGBClassifier(),
              "LGBM": LGBMClassifier(),
              "TabPFN": TabPFNClassifier(),
              }

preproc = ColumnDropper(drop_cols)
# ppl = Pipeline(steps=[('preprocessor', preproc),
#                       ('regressor', model)])
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(missing_values=np.nan, strategy='mean')),
    ('scaler', StandardScaler())
])
results = {}

In [11]:
# run all the models with 10-fold CV, RMSE scoring
for model_name, model in models_bin.items():
    ppl = Pipeline(steps=[('preprocessor', preproc),
                          ('numeric_transformer', numeric_transformer),
                      ('regressor', model)]) # create pipeline object
    ppl.fit(X_train, y_train)
    score = cross_val_score(ppl, X_train, y_train, scoring='roc_auc', cv=3).mean() # 5-fold cv MAE score
    
    print("{}: {:.3f}".format(model_name, score))
    
    results[model_name] = score # record model's performance


Dummy: 0.500
DT: 0.641
XGB: 0.833
[LightGBM] [Info] Number of positive: 2308, number of negative: 789
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001232 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4662
[LightGBM] [Info] Number of data points in the train set: 3097, number of used features: 362
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.745237 -> initscore=1.073370
[LightGBM] [Info] Start training from score 1.073370
[LightGBM] [Info] Number of positive: 1538, number of negative: 526
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002619 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4578
[LightGBM] [Info] Number of data points in the train set: 2064, number of used features: 361
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.745155

/home/roc/.local/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/roc/.local/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 1539, number of negative: 526
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002501 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4568
[LightGBM] [Info] Number of data points in the train set: 2065, number of used features: 361
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.745278 -> initscore=1.073587
[LightGBM] [Info] Start training from score 1.073587
LGBM: 0.843


/home/roc/.local/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


TabPFN: 0.851


In [12]:
score

0.8514077135589445

In [ ]:
# # visualize LGBM's predictions agains y_true on test set
# df_viz = pd.DataFrame({"id":range(len(X_test)), 
#                        "pred":ppl.predict(X_test),
#                       "true": y_test})

# df_viz['pred-true'] = df_viz['pred'] - df_viz['true']

# # melt for viz
# df_viz_melt = df_viz.melt(id_vars='id', 
#            var_name='type',
#             value_vars=['pred', 'true'],
#             value_name='y'
#            )


In [13]:
# visualize LGBM important features
ftr_imptns = ppl['regressor'].feature_importances_

ftr_imptns_df = pd.DataFrame({"feature": ppl['preprocessor'].remaining_cols,
             "importance": ftr_imptns})
ftr_imptns_df = ftr_imptns_df.sort_values('importance', ascending = False).reset_index(drop=True)

n = 30 # how many top features to display in the plot
bars = alt.Chart(ftr_imptns_df.iloc[:n, :]).mark_bar().encode(
    x='importance:Q',
    y=alt.Y('feature:O', sort='-x')
)
bars.properties(
    title="LGBM - feature importance (top30)"
)


AttributeError: 'TabPFNClassifier' object has no attribute 'feature_importances_'